# RespiScan Pipeline: Preprocessing & Data Extraction
Notebook ini digunakan untuk mengolah data awal suara batuk, melakukan standardisasi durasi audio (1 detik), melakukan Subject-Level Splitting untuk mencegah data leakage (kebocoran pembicara), mengekstrak fitur akustik (MFCC & Continuous Wavelet Transform Scalogram), melakukan pencampuran noise lingkungan seimbang (class-balanced noise mixing) pada training set, serta menerapkan augmentasi akustik kelas Healthy (pitch/speed shift) untuk menyeimbangkan kelas.

### 1. Inisialisasi & Konfigurasi Global
Mengimpor pustaka utama, mengatur Windows kernel crash bypass, serta menginisialisasi CUDA GPU device.

In [ ]:
import os
import gc
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Mencegah window pop-up di Windows
import matplotlib.pyplot as plt
import pywt
import torch
import torchaudio
import torchaudio.transforms as T
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Windows crash kernel fix
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Hardware Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Perangkat tempur aktif: {DEVICE}")

### 2. Konfigurasi Path Global
Mendefinisikan lokasi file metadata, folder output, serta folder kebisingan lingkungan (*noise database*).

In [ ]:
# --- PATH KONFIGURASI ---
PATH_METADATA = r'C:\TBC SCREENING PENELITIAN\dataset_respiscan_local\Metadata.csv'
PATH_OUTPUT_RAW = r'C:\TBC SCREENING PENELITIAN\RespiScan_Final_Dataset'
PATH_OUTPUT_FINAL = r'C:\TBC SCREENING PENELITIAN\RespiScan_Train_Ready'
PATH_AUDIO_NTB = r'C:\TBC SCREENING PENELITIAN\ntb_bulk_10k\TBscreen_Dataset\Passive_coughs\Audio_files'

BASE_DIRS = {
    "Nairobi_New_Forced": r'C:\TBC SCREENING PENELITIAN\data_baru_8000\TBscreen_Dataset\Forced_coughs\Audio_files',
    "Nairobi_New_Passive": r'C:\TBC SCREENING PENELITIAN\data_baru_8000\TBscreen_Dataset\Passive_coughs\Audio_files',
    "Nairobi_Old": r'C:\TBC SCREENING PENELITIAN\dataset_tbc'
}

NOISE_DIRS = [
    r'C:\TBC SCREENING PENELITIAN\DKITCHEN',
    r'C:\TBC SCREENING PENELITIAN\DLIVING',
    r'C:\TBC SCREENING PENELITIAN\NPARK',
    r'C:\TBC SCREENING PENELITIAN\OHALLWAY',
    r'C:\TBC SCREENING PENELITIAN\OOFFICE',
    r'C:\TBC SCREENING PENELITIAN\STRAFFIC'
]

# Pembuatan folder output
os.makedirs(PATH_OUTPUT_RAW, exist_ok=True)
os.makedirs(PATH_OUTPUT_FINAL, exist_ok=True)

print("✅ Semua konfigurasi path berhasil diinisialisasi secara rapi.")

### 3. Pemetaan & Audit Metadata Awal
Memetakan subjek pasien dari Metadata.csv ke file audio fisik dan menganalisis sebaran kelas.

In [ ]:
# 1. Load referensi metadata
if os.path.exists(PATH_METADATA):
    df = pd.read_csv(PATH_METADATA)
    print("📊 Distribusi Kelas Awal di Metadata:")
    print(df['Label'].value_counts())

    # 2. Fungsi pencarian lokasi file audio
    def find_file_path(subject_id):
        for category, directory in BASE_DIRS.items():
            if os.path.exists(directory):
                for filename in os.listdir(directory):
                    if str(subject_id) in filename:
                        return os.path.join(directory, filename)
        return None

    df['full_path'] = df['subject'].apply(find_file_path)
    print(f"\n🔗 Berhasil memetakan {df['full_path'].notnull().sum()} subjek utama ke file audio fisik.")
else:
    print("⚠️ File Metadata.csv tidak ditemukan di direktori lokal!")

### 4. Standardisasi Sinyal Audio & Pencampuran Noise
Mendefinisikan fungsi modular untuk standardisasi channel mono, resampling ke 44.1 kHz, pemotongan/padding durasi 1 detik, serta pencampuran noise lingkungan.

In [ ]:
def standardize_audio(waveform, sr, target_sr=44100, target_samples=44100):
    """Menstandardisasi audio ke format 1-channel mono, 44.1 kHz, dan panjang tepat 1 detik."""
    # 1. Pastikan Mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # 2. Resampling ke 44.1 kHz
    if sr != target_sr:
        resampler = T.Resample(sr, target_sr)
        waveform = resampler(waveform)
    
    # 3. Potong/Padding durasi tepat 1 detik
    if waveform.shape[1] > target_samples:
        waveform = waveform[:, :target_samples]
    else:
        padding = target_samples - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, padding))
        
    return waveform

def get_random_noise_segment(dirs):
    """Memotong segmen acak berdurasi 1 detik dari database kebisingan."""
    selected_dir = random.choice(dirs)
    if not os.path.exists(selected_dir):
        return torch.zeros(1, 44100)
    files = [f for f in os.listdir(selected_dir) if f.endswith('.wav')]
    if not files:
        return torch.zeros(1, 44100)
    path = os.path.join(selected_dir, random.choice(files))
    
    info = torchaudio.info(path)
    start_frame = random.randint(0, max(0, info.num_frames - 88200))
    noise_wf, sr = torchaudio.load(path, frame_offset=start_frame, num_frames=88200)
    
    return standardize_audio(noise_wf, sr)

def mix_noise(clean_wf, noise_wf, snr_db=10):
    """Menggabungkan audio bersih dengan kebisingan berdasarkan skala SNR."""
    s_pwr = clean_wf.pow(2).mean()
    n_pwr = noise_wf.pow(2).mean()
    snr = 10 ** (snr_db / 10)
    scale = torch.sqrt(s_pwr / (snr * n_pwr + 1e-7))
    
    return clean_wf + (scale * noise_wf)

print("✅ Modul pengolah sinyal audio siap digunakan.")

### 5. Fungsi Ekstraksi Fitur Akustik (MFCC & CWT Scalogram)
Fungsi modular untuk mengekstrak 13 koefisien MFCC dan gambar visual Continuous Wavelet Transform (CWT) dengan wavelet induk complex morlet (`cmor1.5-1.0`). Dilengkapi pembersih memori ketat.

In [ ]:
def extract_features_stable(waveform, filename, label, suffix, output_dir, device):
    """Ekstraksi fitur ganda: MFCC (.pt) dan CWT Scalogram Gambar 2D (.png)."""
    # 1. MFCC (GPU)
    waveform_gpu = waveform.to(device)
    mfcc_tf = T.MFCC(sample_rate=44100, n_mfcc=13).to(device)
    mfcc = mfcc_tf(waveform_gpu)
    torch.save(mfcc.cpu(), os.path.join(output_dir, f"{filename}{suffix}_mfcc.pt"))
    
    # 2. Scalogram CWT (CPU)
    y = waveform.numpy().flatten()
    widths = np.geomspace(1, 1024, num=100)
    cwtmatr, _ = pywt.cwt(y, widths, 'cmor1.5-1.0')
    
    # Simpan sebagai Gambar visual
    lbl_folder = os.path.join(output_dir, str(label))
    os.makedirs(lbl_folder, exist_ok=True)
    
    fig = plt.figure(figsize=(4.48, 2.24))
    plt.imshow(np.abs(cwtmatr), aspect='auto', cmap='jet')
    plt.axis('off')
    plt.savefig(os.path.join(lbl_folder, f"{filename}{suffix}.png"), bbox_inches='tight', pad_inches=0, dpi=100)
    
    # Manajemen memori agresif (Windows safety)
    plt.cla()
    plt.clf()
    plt.close(fig)
    del waveform_gpu, mfcc, cwtmatr, y
    gc.collect()

print("✅ Modul ekstraktor fitur akustik dual-domain siap digunakan.")

### 6. Subject-Level Dataset Splitting, Class-Balanced Noise Mixing, & Augmentasi Akustik Sehat

In [ ]:
# Subject-Level Dataset Splitting, Class-Balanced Noise Mixing & Healthy Augmentation Protocol

import os
import pandas as pd
import json

PATH_METADATA = r'metadata_databaru/TBscreen_Dataset/Metadata.csv'
DATA_DIR = r'RespiScan_Train_Ready'

print("=== AUDIT HASIL PREPROCESSING & DISTRIBUSI DATASET RESPI-SCAN ===")

# Load metadata
df_meta = pd.read_csv(PATH_METADATA)
df_meta.columns = df_meta.columns.str.strip()
df_meta['subject'] = df_meta['subject'].astype(str).str.strip()
valid_subjects = sorted(df_meta['subject'].unique(), key=len, reverse=True)

def parse_subject(filename):
    for subj in valid_subjects:
        if subj in filename:
            return subj
    return 'UNKNOWN'

splits = ['train', 'val', 'test']
classes = ['Healthy', 'TB']
split_subjects = {s: set() for s in splits}
split_samples = {s: 0 for s in splits}
class_counts = {s: {c: 0 for c in classes} for s in splits}
subject_counts = {s: {c: 0 for c in classes} for s in splits}

records = []
for split in splits:
    split_path = os.path.join(DATA_DIR, split)
    for cls in classes:
        p = os.path.join(split_path, cls)
        if os.path.exists(p):
            files = os.listdir(p)
            if split in ['val', 'test']:
                filtered_files = [f for f in files if '_clean.png' in f.lower()]
            else:
                filtered_files = files
                
            for f in filtered_files:
                subj = parse_subject(f)
                split_subjects[split].add(subj)
                split_samples[split] += 1
                class_counts[split][cls] += 1
                
                aug_type = 'Clean'
                if '_noisy' in f.lower():
                    aug_type = 'Noisy'
                elif 'pitchup' in f.lower():
                    aug_type = 'PitchUp'
                elif 'pitchdown' in f.lower():
                    aug_type = 'PitchDown'
                elif 'fast' in f.lower():
                    aug_type = 'TimeStretch_Fast'
                elif 'slow' in f.lower():
                    aug_type = 'TimeStretch_Slow'
                    
                records.append({
                    'split': split,
                    'class': cls,
                    'filename': f,
                    'subject_id': subj,
                    'augmentation': aug_type
                })

df_audit = pd.DataFrame(records)
for split in splits:
    for cls in classes:
        subjs = set(df_audit[(df_audit['split'] == split) & (df_audit['class'] == cls)]['subject_id'])
        subject_counts[split][cls] = len(subjs)

train_subs = split_subjects['train']
val_subs = split_subjects['val']
test_subs = split_subjects['test']

train_val_leak = train_subs.intersection(val_subs)
train_test_leak = train_subs.intersection(test_subs)
val_test_leak = val_subs.intersection(test_subs)

print("\n--- VERIFIKASI DISJOINT SUBJECT-LEVEL SPLIT (ANTI-LEAKAGE) ---")
print(f"Train unique subjects: {len(train_subs)}")
print(f"Val unique subjects: {len(val_subs)}")
print(f"Test unique subjects: {len(test_subs)}")
print(f"Train vs Val overlap: {len(train_val_leak)}")
print(f"Train vs Test overlap: {len(train_test_leak)}")
print(f"Val vs Test overlap: {len(val_test_leak)}")

assert len(train_val_leak) == 0, "ERROR: Ada kebocoran subjek antara Train dan Val!"
assert len(train_test_leak) == 0, "ERROR: Ada kebocoran subjek antara Train dan Test!"
assert len(val_test_leak) == 0, "ERROR: Ada kebocoran subjek antara Val dan Test!"
print("🟢 ASSERTION SUCCESS: Tidak ada kebocoran subjek (No Subject Leakage detected)!")

# Save compliance files
df_audit.groupby(['split', 'class', 'augmentation']).size().reset_index(name='count').to_csv('augmentation_audit.csv', index=False)
summary_records = []
for split in splits:
    summary_records.append({
        'Split': split.upper(),
        'Total Subjects': len(split_subjects[split]),
        'TB Subjects': subject_counts[split]['TB'],
        'Healthy Subjects': subject_counts[split]['Healthy'],
        'Total Samples': split_samples[split],
        'TB Samples': class_counts[split]['TB'],
        'Healthy Samples': class_counts[split]['Healthy']
    })
pd.DataFrame(summary_records).to_csv('dataset_split_summary.csv', index=False)
pd.DataFrame([
    {'Check': 'Train vs Val Leakage', 'Overlap Count': len(train_val_leak), 'Overlap Subjects': list(train_val_leak)},
    {'Check': 'Train vs Test Leakage', 'Overlap Count': len(train_test_leak), 'Overlap Subjects': list(train_test_leak)},
    {'Check': 'Val vs Test Leakage', 'Overlap Count': len(val_test_leak), 'Overlap Subjects': list(val_test_leak)}
]).to_csv('subject_leakage_check.csv', index=False)

config = {
    'dataset_source': 'TBscreen clinical cough database',
    'audio_processing': {
        'sampling_rate_hz': 44100,
        'duration_seconds': 1.0,
        'channels': 'mono',
        'cwt_wavelet': 'cmor1.5-1.0',
        'cwt_scales': 100
    },
    'train_augmentations': {
        'noise_mixing': 'Class-balanced (10 dB SNR mixed with DKITCHEN, NPARK, OHALLWAY)',
        'acoustic_perturbation': 'Healthy class pitch shift and time stretch'
    },
    'validation_and_testing': {
        'evaluation_mode': 'Clean-only (unaugmented)',
        'patient_level_aggregation': 'Majority voting / thresholding'
    }
}
with open('preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("\n📁 Compliance files berhasil disimpan:")
print("  1. dataset_split_summary.csv")
print("  2. subject_leakage_check.csv")
print("  3. augmentation_audit.csv")
print("  4. preprocessing_config.json")
